# 02. Feature Engineering

### 피처 생성 순서
1. processed 데이터 로드
2. 항만별 피벗 테이블 생성
3. 시간 피처 (요일, 월, 공휴일 등)
4. 항만 피처 + 선박유형 비율 피처
5. 래그 피처 (전주 동기간 대비 변화율)
6. 연쇄 혼잡 lag 피처 (EDA 결과: 부산→울산 2일, 나머지 1일)
7. 슬라이딩 윈도우 데이터셋 생성 (입력 30일 → 예측 7일)
8. **분할 후** MinMaxScaler (train-only fit → leakage 방지)
9. 저장

---
## 1. 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

PROCESSED = Path('../data/processed')

vessel = pd.read_csv(PROCESSED / 'vessel_clean.csv',
                     parse_dates=['입항일시', '출항일시', 'date'],
                     encoding='utf-8-sig')
daily = pd.read_csv(PROCESSED / 'daily_aggregated.csv',
                    parse_dates=['date'],
                    encoding='utf-8-sig')

print(f'vessel_clean  : {len(vessel):,}건')
print(f'daily_aggregated: {len(daily):,}건')
daily.head()

/var/folders/ng/g2w4ytb93_x85wywst5vv0100000gn/T/ipykernel_95911/2271015783.py:11: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  vessel = pd.read_csv(PROCESSED / 'vessel_clean.csv',


vessel_clean  : 352,365건
daily_aggregated: 1,407건


,항명,date,입항수,평균체류시간,체선율,컨테이너_비율,유조선_비율,벌크_비율,일반화물_비율
0,광양,2024-06-20,62,39.075806,0.129032,0.387097,0.032258,0.193548,0.161290
1,광양,2024-06-21,92,39.724275,0.021739,0.369565,0.108696,0.086957,0.282609
2,광양,2024-06-22,64,35.117708,0.125000,0.343750,0.156250,0.156250,0.156250
3,광양,2024-06-23,48,20.666667,0.000000,0.333333,0.208333,0.166667,0.125000
4,광양,2024-06-24,68,29.476961,0.058824,0.205882,0.205882,0.147059,0.147059


---
## 2. 항만별 피벗 테이블 생성

In [2]:
TARGET_PORTS = ['부산', '울산', '인천', '광양']
SHIP_GROUPS = ['컨테이너', '유조선', '벌크', '일반화물']

# 날짜 전체 범위 생성 (결측일 채우기)
date_range = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')

def pivot_metric(metric_col, fill=0):
    pv = daily.pivot_table(index='date', columns='항명', values=metric_col)
    pv = pv.reindex(date_range).ffill()
    return pv.fillna(fill)

pv_rate  = pivot_metric('체선율')
pv_count = pivot_metric('입항수')
pv_stay  = pivot_metric('평균체류시간')

# 선박유형 비율 피벗
pv_ship = {}
for g in SHIP_GROUPS:
    col = f'{g}_비율'
    if col in daily.columns:
        pv_ship[g] = pivot_metric(col)

print('체선율 피벗:')
print(pv_rate.tail(3))
print(f'\n선박유형 피벗 생성: {list(pv_ship.keys())}')

체선율 피벗:
항명                광양        부산        울산        인천
2026-05-17  0.019608  0.000000  0.038462  0.050847
2026-05-18  0.000000  0.031646  0.000000  0.067797
2026-05-19  0.000000  0.024793  0.019608  0.022222

선박유형 피벗 생성: ['컨테이너', '유조선', '벌크', '일반화물']


---
## 3. 시간 피처

In [3]:
df_feat = pd.DataFrame(index=date_range)
df_feat.index.name = 'date'

df_feat['dayofweek'] = df_feat.index.dayofweek        # 0=월요일
df_feat['month']     = df_feat.index.month
df_feat['quarter']   = df_feat.index.quarter
df_feat['is_weekend'] = (df_feat.index.dayofweek >= 5).astype(int)

# 주요 공휴일 (설·추석 연휴는 체선에 큰 영향)
# 2024~2026년 주요 공휴일 날짜
holidays = [
    '2024-09-14', '2024-09-15', '2024-09-16', '2024-09-17', '2024-09-18',  # 추석
    '2025-01-28', '2025-01-29', '2025-01-30',  # 설날
    '2025-10-02', '2025-10-03', '2025-10-04', '2025-10-05', '2025-10-06',  # 추석
    '2026-02-16', '2026-02-17', '2026-02-18',  # 설날
]
df_feat['is_holiday'] = df_feat.index.isin(pd.to_datetime(holidays)).astype(int)

print('시간 피처 샘플:')
print(df_feat.head(10))

시간 피처 샘플:
            dayofweek  month  quarter  is_weekend  is_holiday
date                                                         
2024-06-20          3      6        2           0           0
2024-06-21          4      6        2           0           0
2024-06-22          5      6        2           1           0
2024-06-23          6      6        2           1           0
2024-06-24          0      6        2           0           0
2024-06-25          1      6        2           0           0
2024-06-26          2      6        2           0           0
2024-06-27          3      6        2           0           0
2024-06-28          4      6        2           0           0
2024-06-29          5      6        2           1           0


--- 
## 4. 항만 피처 + 선박유형 비율 추가

In [4]:
for port in TARGET_PORTS:
    if port in pv_rate.columns:
        df_feat[f'{port}_체선율']   = pv_rate[port]
        df_feat[f'{port}_입항수']   = pv_count[port]
        df_feat[f'{port}_평균체류'] = pv_stay[port]

# 선박유형 비율 피처 추가
for g, pv in pv_ship.items():
    for port in TARGET_PORTS:
        if port in pv.columns:
            df_feat[f'{port}_{g}비율'] = pv[port]

print(f'시간 + 항만 + 선박유형 피처 합계: {df_feat.shape[1]}개')
df_feat.head(3)

시간 + 항만 + 선박유형 피처 합계: 33개


,dayofweek,month,quarter,is_weekend,is_holiday,부산_체선율,부산_입항수,부산_평균체류,울산_체선율,울산_입항수,...,인천_유조선비율,광양_유조선비율,부산_벌크비율,울산_벌크비율,인천_벌크비율,광양_벌크비율,부산_일반화물비율,울산_일반화물비율,인천_일반화물비율,광양_일반화물비율
date,,,,,,,,,,,,,,,,,,,,,
2024-06-20,3,6,2,0,0,0.145038,262.0,65.966412,0.073529,136.0,...,0.224138,0.032258,0.045802,0.014706,0.034483,0.193548,0.068702,0.014706,0.155172,0.161290
2024-06-21,4,6,2,0,0,0.097015,268.0,55.335075,0.140625,128.0,...,0.195652,0.108696,0.022388,0.046875,0.065217,0.086957,0.059701,0.093750,0.217391,0.282609
2024-06-22,5,6,2,1,0,0.084746,236.0,46.741667,0.050000,120.0,...,0.228571,0.156250,0.033898,0.050000,0.085714,0.156250,0.025424,0.083333,0.171429,0.156250


---
## 5. 래그 피처 (7일, 14일, 30일 전 체선율)

In [5]:
for port in TARGET_PORTS:
    col = f'{port}_체선율'
    if col not in df_feat.columns:
        continue
    for lag in [7, 14, 30]:
        df_feat[f'{port}_lag{lag}'] = df_feat[col].shift(lag)

    # 7일 이동평균
    df_feat[f'{port}_ma7']  = df_feat[col].shift(1).rolling(7).mean()
    df_feat[f'{port}_ma14'] = df_feat[col].shift(1).rolling(14).mean()

print('래그 피처 포함 컬럼 수:', df_feat.shape[1])

래그 피처 포함 컬럼 수: 53


---
## 6. 연쇄 혼잡 lag 피처

> EDA 결과 (CCF):
> - 부산 → 울산: 2일 후 전이
> - 부산 → 광양: 1일 후 전이
> - 부산 → 인천: 1일 후 전이
> - 울산 → 광양: 1일 후 전이

In [6]:
# EDA Cross-correlation 결과 기반 lag 설정
CASCADE_LAGS = [
    ('부산', '울산', 2),
    ('부산', '광양', 1),
    ('부산', '인천', 1),
    ('울산', '광양', 1),
]

for src, dst, lag in CASCADE_LAGS:
    src_col = f'{src}_체선율'
    if src_col in df_feat.columns:
        df_feat[f'{src}_to_{dst}_lag{lag}'] = df_feat[src_col].shift(lag)

print('연쇄 혼잡 lag 피처 추가 후 컬럼 수:', df_feat.shape[1])
cascade_cols = [c for c in df_feat.columns if 'to_' in c]
print('추가된 피처:', cascade_cols)

연쇄 혼잡 lag 피처 추가 후 컬럼 수: 57
추가된 피처: ['부산_to_울산_lag2', '부산_to_광양_lag1', '부산_to_인천_lag1', '울산_to_광양_lag1']


---
## 7. 슬라이딩 윈도우 데이터셋 생성

In [7]:
# NaN이 있는 초기 행 제거 (lag 30일 필요)
df_feat = df_feat.dropna().reset_index()
print(f'NaN 제거 후: {len(df_feat)}일치 데이터')

INPUT_WINDOW = 30   # 입력: 과거 30일
PRED_HORIZON = 7    # 예측: 향후 7일

# 피처 컬럼 (date 제외)
feature_cols = [c for c in df_feat.columns if c != 'date']

# 타겟 컬럼: 각 항만의 체선율
target_cols = [f'{p}_체선율' for p in TARGET_PORTS if f'{p}_체선율' in df_feat.columns]

X_list, y_list, date_list = [], [], []

for i in range(INPUT_WINDOW, len(df_feat) - PRED_HORIZON + 1):
    x_window = df_feat[feature_cols].iloc[i - INPUT_WINDOW:i].values  # (30, n_feat)
    y_window = df_feat[target_cols].iloc[i:i + PRED_HORIZON].values    # (7, n_port)
    X_list.append(x_window)
    y_list.append(y_window)
    date_list.append(df_feat['date'].iloc[i])

X = np.array(X_list, dtype=np.float32)  # (N, 30, n_feat)
y = np.array(y_list, dtype=np.float32)  # (N, 7, n_port)

print(f'X shape: {X.shape}  (샘플 수, 윈도우, 피처)')
print(f'y shape: {y.shape}  (샘플 수, horizon, 항만 수)')
print(f'피처 컬럼 수: {len(feature_cols)}')
print(f'타겟 항만 수: {len(target_cols)} → {target_cols}')

NaN 제거 후: 669일치 데이터


X shape: (633, 30, 57)  (샘플 수, 윈도우, 피처)
y shape: (633, 7, 4)  (샘플 수, horizon, 항만 수)
피처 컬럼 수: 57
타겟 항만 수: 4 → ['부산_체선율', '울산_체선율', '인천_체선율', '광양_체선율']


---
## 8. 학습/검증/테스트 분할 후 MinMaxScaler (train-only fit)

> **수정**: 분할을 먼저 하고 X_train에만 scaler fit → val/test leakage 방지

In [8]:
from sklearn.preprocessing import MinMaxScaler
import pickle

# ① 분할 먼저
n = len(X)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)

X_train_raw = X[:n_train]
X_val_raw   = X[n_train:n_train+n_val]
X_test_raw  = X[n_train+n_val:]
y_train_raw = y[:n_train]
y_val_raw   = y[n_train:n_train+n_val]
y_test_raw  = y[n_train+n_val:]

# ② X: train에만 fit
n_feats = X.shape[2]
scaler_X = MinMaxScaler()
scaler_X.fit(X_train_raw.reshape(-1, n_feats))

X_train = scaler_X.transform(X_train_raw.reshape(-1, n_feats)).reshape(X_train_raw.shape).astype('float32')
X_val   = scaler_X.transform(X_val_raw.reshape(-1, n_feats)).reshape(X_val_raw.shape).astype('float32')
X_test  = scaler_X.transform(X_test_raw.reshape(-1, n_feats)).reshape(X_test_raw.shape).astype('float32')

# ③ y: train에만 fit
n_ports = y.shape[2]
scaler_y = MinMaxScaler()
scaler_y.fit(y_train_raw.reshape(-1, n_ports))

y_train = scaler_y.transform(y_train_raw.reshape(-1, n_ports)).reshape(y_train_raw.shape).astype('float32')
y_val   = scaler_y.transform(y_val_raw.reshape(-1, n_ports)).reshape(y_val_raw.shape).astype('float32')
y_test  = scaler_y.transform(y_test_raw.reshape(-1, n_ports)).reshape(y_test_raw.shape).astype('float32')

print(f'Train: X{X_train.shape}  y{y_train.shape}')
print(f'Val  : X{X_val.shape}  y{y_val.shape}')
print(f'Test : X{X_test.shape}  y{y_test.shape}')
print(f'X_val 범위 : [{X_val.min():.4f}, {X_val.max():.4f}]  (train-only fit 시 1 초과 가능)')
print(f'X_test 범위: [{X_test.min():.4f}, {X_test.max():.4f}]')

Train: X(506, 30, 57)  y(506, 7, 4)
Val  : X(63, 30, 57)  y(63, 7, 4)
Test : X(64, 30, 57)  y(64, 7, 4)
X_val 범위 : [0.0000, 1.0000]  (train-only fit 시 1 초과 가능)
X_test 범위: [-0.0360, 2.0000]


---
## 9. 저장

In [9]:
# ⚠️ 이 셀은 실행하지 마세요.
# 피처 생성 파이프라인이 scripts/add_weather_features.py로 이전되었습니다.
# 해당 스크립트가 날씨 피처(85개) + 이벤트 피처(93개)를 포함한 올바른 버전을 생성합니다.
# python scripts/add_weather_features.py
raise RuntimeError("이 셀 대신 scripts/add_weather_features.py 를 실행하세요.")

PROCESSED = Path('../data/processed')

np.save(PROCESSED / 'X_train.npy', X_train)
np.save(PROCESSED / 'y_train.npy', y_train)
np.save(PROCESSED / 'X_val.npy',   X_val)
np.save(PROCESSED / 'y_val.npy',   y_val)
np.save(PROCESSED / 'X_test.npy',  X_test)
np.save(PROCESSED / 'y_test.npy',  y_test)

with open(PROCESSED / 'scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open(PROCESSED / 'scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

import json as _json
meta = {
    'feature_cols': feature_cols,
    'target_cols':  target_cols,
    'input_window': INPUT_WINDOW,
    'pred_horizon': PRED_HORIZON,
    'n_train': int(len(X_train)),
    'n_val':   int(len(X_val)),
    'n_test':  int(len(X_test)),
    'cascade_lags': [{'src': s, 'dst': d, 'lag': l} for s, d, l in CASCADE_LAGS],
    'scaler_fit': 'train_only',
}
with open(PROCESSED / 'dataset_meta.json', 'w', encoding='utf-8') as f:
    _json.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 완료:')
for fname in ['X_train.npy','y_train.npy','X_val.npy','y_val.npy','X_test.npy','y_test.npy']:
    p = PROCESSED / fname
    print(f'  {fname}: {p.stat().st_size/1024:.1f} KB')
print(f'\nscaler_fit: train_only ✅')
print(f'피처 수: {len(feature_cols)}')

저장 완료:
  X_train.npy: 3380.0 KB
  y_train.npy: 55.5 KB
  X_val.npy: 420.9 KB
  y_val.npy: 7.0 KB
  X_test.npy: 427.6 KB
  y_test.npy: 7.1 KB

scaler_fit: train_only ✅
피처 수: 57


--- 
## 10. 이벤트 피처 통합 (LLM Agent 분류 결과)

> **NOTE**: `data/processed/event_flags.csv`는 LLM Agent(Claude claude-haiku-4-5) 분류 결과를 사용합니다.  
> API key 설정 후 `python scripts/classify_events.py` 실행 시 실제 분류값으로 교체됩니다.  
> 현재는 규칙 기반 placeholder를 사용합니다.

In [10]:
# 이벤트 플래그 로드
event_path = PROCESSED / "event_flags.csv"
df_events = pd.read_csv(event_path, parse_dates=["date"]).set_index("date")
EVENT_COLS = ["strike", "weather", "surge", "any_event"]
print(f"이벤트 플래그: {df_events.shape}")
print(df_events[EVENT_COLS].sum())
df_events.head(3)

이벤트 플래그: (699, 4)
strike       2
weather      0
surge        3
any_event    5
dtype: int64


,strike,weather,surge,any_event
date,,,,
2024-06-20,0,0,0,0
2024-06-21,0,0,0,0
2024-06-22,0,0,0,0


In [11]:
# df_feat에 이벤트 피처 병합
# df_feat는 Cell 14에서 dropna().reset_index()된 상태 → date 컬럼 존재
df_feat_event = df_feat.merge(
    df_events[EVENT_COLS].reset_index(),
    on="date", how="left"
).fillna(0)  # 이벤트 없는 날 → 0

feature_cols_event = feature_cols + EVENT_COLS
N_FEATURES_EVENT = len(feature_cols_event)
print(f"피처 수: {N_FEATURES_EVENT} (기존 {len(feature_cols)} + 이벤트 {len(EVENT_COLS)})")
df_feat_event[EVENT_COLS].sum()

피처 수: 61 (기존 57 + 이벤트 4)


strike       2
weather      0
surge        3
any_event    5
dtype: int64

In [12]:
# 이벤트 포함 슬라이딩 윈도우 생성
INPUT_WINDOW = 30
PRED_HORIZON = 7
X_ev_list, y_ev_list = [], []

for i in range(INPUT_WINDOW, len(df_feat_event) - PRED_HORIZON + 1):
    X_ev_list.append(df_feat_event[feature_cols_event].iloc[i - INPUT_WINDOW:i].values)
    y_ev_list.append(df_feat_event[target_cols].iloc[i:i + PRED_HORIZON].values)

X_ev = np.array(X_ev_list, dtype=np.float32)
y_ev = np.array(y_ev_list, dtype=np.float32)
print(f"X_ev: {X_ev.shape}, y_ev: {y_ev.shape}")

X_ev: (633, 30, 61), y_ev: (633, 7, 4)


In [13]:
# 분할 + 스케일링 (train-only fit)
n_ev = len(X_ev)
n_ev_train = int(n_ev * 0.8)
n_ev_val   = int(n_ev * 0.1)

X_ev_train_raw = X_ev[:n_ev_train]
X_ev_val_raw   = X_ev[n_ev_train:n_ev_train+n_ev_val]
X_ev_test_raw  = X_ev[n_ev_train+n_ev_val:]
y_ev_train_raw = y_ev[:n_ev_train]
y_ev_val_raw   = y_ev[n_ev_train:n_ev_train+n_ev_val]
y_ev_test_raw  = y_ev[n_ev_train+n_ev_val:]

n_ev_f = X_ev_train_raw.shape[2]
scaler_X_ev = MinMaxScaler()
scaler_X_ev.fit(X_ev_train_raw.reshape(-1, n_ev_f))

def scale_X(raw, sc, n_f):
    sh = raw.shape
    return sc.transform(raw.reshape(-1, n_f)).reshape(sh).astype("float32")

X_ev_train = scale_X(X_ev_train_raw, scaler_X_ev, n_ev_f)
X_ev_val   = scale_X(X_ev_val_raw,   scaler_X_ev, n_ev_f)
X_ev_test  = scale_X(X_ev_test_raw,  scaler_X_ev, n_ev_f)

# y 스케일러는 기존 scaler_y 재사용 (같은 타겟)
n_p = y_ev_train_raw.shape[2]
scaler_y_ev = MinMaxScaler()
scaler_y_ev.fit(y_ev_train_raw.reshape(-1, n_p))

def scale_y(raw, sc, n_p):
    sh = raw.shape
    return sc.transform(raw.reshape(-1, n_p)).reshape(sh).astype("float32")

y_ev_train = scale_y(y_ev_train_raw, scaler_y_ev, n_p)
y_ev_val   = scale_y(y_ev_val_raw,   scaler_y_ev, n_p)
y_ev_test  = scale_y(y_ev_test_raw,  scaler_y_ev, n_p)

print(f"Train: X={X_ev_train.shape}, y={y_ev_train.shape}")
print(f"Val:   X={X_ev_val.shape},   y={y_ev_val.shape}")
print(f"Test:  X={X_ev_test.shape},  y={y_ev_test.shape}")

Train: X=(506, 30, 61), y=(506, 7, 4)
Val:   X=(63, 30, 61),   y=(63, 7, 4)
Test:  X=(64, 30, 61),  y=(64, 7, 4)


In [14]:
# 이벤트 포함 데이터셋 저장
for name, arr in [("X_ev_train", X_ev_train), ("y_ev_train", y_ev_train),
                  ("X_ev_val",   X_ev_val),   ("y_ev_val",   y_ev_val),
                  ("X_ev_test",  X_ev_test),  ("y_ev_test",  y_ev_test)]:
    np.save(PROCESSED / f"{name}.npy", arr)

with open(PROCESSED / "scaler_X_ev.pkl", "wb") as f:
    pickle.dump(scaler_X_ev, f)
with open(PROCESSED / "scaler_y_ev.pkl", "wb") as f:
    pickle.dump(scaler_y_ev, f)

import json as _json
meta_ev = {
    "feature_cols": feature_cols_event,
    "target_cols":  target_cols,
    "n_features":   N_FEATURES_EVENT,
    "n_event_features": len(EVENT_COLS),
    "input_window": INPUT_WINDOW,
    "pred_horizon": PRED_HORIZON,
    "scaler_fit":   "train_only",
    "n_train": int(len(X_ev_train)),
    "n_val":   int(len(X_ev_val)),
    "n_test":  int(len(X_ev_test)),
}
with open(PROCESSED / "dataset_ev_meta.json", "w", encoding="utf-8") as f:
    _json.dump(meta_ev, f, ensure_ascii=False, indent=2)

print("이벤트 포함 데이터셋 저장 완료")
print(f"  피처: {N_FEATURES_EVENT}개 ({len(feature_cols)}개 수치 + {len(EVENT_COLS)}개 이벤트)")

이벤트 포함 데이터셋 저장 완료
  피처: 61개 (57개 수치 + 4개 이벤트)
